# 🚀 Vecna / AIC51: Trích xuất Keyframes & Qwen-VL cho Batch N081-N100

Pipeline tự động hoá hoàn toàn trên Google Colab Pro+:
1. **Tải tốc độ cao (Aria2c)** 2 batch video `.mov` từ server:
   - `https://aic-data.ledo.io.vn/Video_N081-N090.zip` (30 video)
   - `https://aic-data.ledo.io.vn/Video_N091-N100.zip` (30 video)
2. **Chuyển đổi toàn bộ MOV sang MP4**: Dùng ffmpeg remux stream `-c copy` (giữ nguyên 100% chất lượng gốc, frame rate, siêu nhanh và xoá ngay file MOV để tiết kiệm đĩa).
3. **Trích xuất Keyframes chuẩn AIC với cờ `-min 1.0`**: Dùng `aic51-cli add <dir> -d -k -min 1.0` (giãn cách tối thiểu giữa 2 keyframe liên tiếp là 1.0 giây). Xoá ngay MP4 sau khi trích xuất xong.
4. **Sao lưu toàn bộ Keyframes lên Google Drive** (`keyframes_N081_N100.tar.gz`).
5. **Trích xuất vector ngữ nghĩa Qwen-VL** bằng GPU (`qwen_vl.npy`).
6. **Sao lưu toàn bộ Numpy Qwen lên Google Drive** (`features_qwen_N081_N100.tar.gz`) và kiểm tra dữ liệu mẫu.

### 1. Mount Google Drive & Kiểm tra GPU

In [ ]:
# 1. Mount Google Drive để sao lưu kết quả trực tiếp
from google.colab import drive
drive.mount('/content/drive')

# Kiểm tra GPU được cấp (Khuyến nghị chọn A100 / L4 trên Colab Pro+)
!nvidia-smi

### 2. Cài đặt Dependencies Hệ thống & Clone Repo Vecna

In [ ]:
%%bash
# Cài đặt aria2 và ffmpeg để tải và xử lý video
apt-get update -qq
apt-get install -y -qq aria2 ffmpeg unrar

# Clone source code Vecna
cd /content
if [ ! -d "/content/Vecna" ]; then
    git clone https://github.com/nlmhoagn/Vecna.git /content/Vecna
fi

# Cài đặt aic51 CLI từ source
cd /content/Vecna/aic51-src
pip install -q -e .

# Cài đặt thư viện chạy Qwen-VL Embedding
pip install -q sentence-transformers accelerate open_clip_torch deep-translator pymilvus

### 3. Khởi tạo Workspace Vecna & Cấu hình

In [ ]:
%%bash
mkdir -p /content/workspace
cd /content/workspace

# Khởi tạo layout workspace
aic51-cli init

# Copy config.yaml chuẩn vào workspace
if [ -f "/content/Vecna/workspace_2/config.yaml" ]; then
    cp /content/Vecna/workspace_2/config.yaml /content/workspace/config.yaml
elif [ -f "/content/Vecna/config.yaml" ]; then
    cp /content/Vecna/config.yaml /content/workspace/config.yaml
fi
echo "[✓] Đã khởi tạo workspace thành công!"

### 4. Tải 2 Batch Video, Chuyển Đổi MOV sang MP4 & Trích Xuất Keyframes (`-min 1.0`)

- **Quy trình chuẩn hoá từng video**:
  1. Tải zip bằng Aria2c (16 kết nối) & giải nén.
  2. Quét toàn bộ video `.mov`, chuyển đổi sang `.mp4` bằng ffmpeg (`-c copy` siêu nhanh, giữ nguyên 100% chất lượng gốc và frame rate).
  3. Xoá ngay file `.mov` gốc để tiết kiệm đĩa cứng.
  4. Chạy `aic51-cli add <dir> -d -k -min 1.0` để trích xuất keyframes (khoảng cách tối thiểu giữa 2 keyframe liên tiếp là 1.0 giây).
  5. Xoá toàn bộ video `.mp4` để giải phóng bộ nhớ Colab trước khi sang batch tiếp theo.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path
from datetime import datetime

workspace_dir = "/content/workspace"
temp_download = "/content/temp_download"
temp_videos = "/content/temp_videos"
os.makedirs(temp_download, exist_ok=True)
os.makedirs(temp_videos, exist_ok=True)

batches = [
    ("Video_N081-N090.zip", "https://aic-data.ledo.io.vn/Video_N081-N090.zip"),
    ("Video_N091-N100.zip", "https://aic-data.ledo.io.vn/Video_N091-N100.zip")
]

for filename, url in batches:
    zip_path = os.path.join(temp_download, filename)
    print("\n" + "=" * 65)
    print(f"[{datetime.now().strftime('%H:%M:%S')}] ⬇️ 1. Đang tải {filename} qua Aria2c...")
    subprocess.run([
        "aria2c", "-x", "16", "-s", "16", "-k", "1M",
        "--header=User-Agent: Mozilla/5.0",
        "-d", temp_download, "-o", filename, url
    ], check=True)
    
    print(f"[{datetime.now().strftime('%H:%M:%S')}] 📂 2. Đang giải nén {filename}...")
    subprocess.run(["unzip", "-q", "-o", zip_path, "-d", temp_videos], check=True)
    os.remove(zip_path)
    
    # Tìm các file .mov cần chuyển đổi sang .mp4
    mov_files = sorted(list(Path(temp_videos).rglob("*.mov")))
    if mov_files:
        print(f"[{datetime.now().strftime('%H:%M:%S')}] 🔄 3. Đang chuyển đổi {len(mov_files)} video MOV sang MP4...")
        for idx, mov in enumerate(mov_files, 1):
            mp4_target = mov.with_suffix(".mp4")
            # Dùng ffmpeg remux container -c copy (siêu nhanh, giữ nguyên 100% frame/fps)
            cmd = ["ffmpeg", "-y", "-i", str(mov), "-c", "copy", "-map", "0:v", "-map", "0:a?", str(mp4_target)]
            res = subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
            if res.returncode != 0 or not mp4_target.exists() or mp4_target.stat().st_size == 0:
                # Fallback encode nếu remux không tương thích
                subprocess.run([
                    "ffmpeg", "-y", "-i", str(mov),
                    "-c:v", "libx264", "-preset", "veryfast", "-crf", "18",
                    "-c:a", "aac", str(mp4_target)
                ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            # Xoá file .mov ngay để tiết kiệm dung lượng đĩa
            mov.unlink()
            if idx % 10 == 0 or idx == len(mov_files):
                print(f"   - Đã chuyển đổi: {idx}/{len(mov_files)} video sang MP4")
    
    # Quét danh sách video MP4 để trích xuất keyframes
    mp4_files = sorted(list(Path(temp_videos).rglob("*.mp4")))
    if not mp4_files:
        raise FileNotFoundError(f"Không tìm thấy video .mp4 nào trong {filename}")
    
    video_dir = mp4_files[0].parent
    print(f"[{datetime.now().strftime('%H:%M:%S')}] 🎬 4. Đang trích xuất keyframes với cờ -min 1.0 ({len(mp4_files)} video tại {video_dir})...")
    
    # Chạy aic51-cli add với cờ -min 1.0 theo yêu cầu
    subprocess.run([
        "aic51-cli", "add", str(video_dir), "-d", "-k", "-min", "1.0"
    ], cwd=workspace_dir, check=True)
    
    # Xoá video mp4 sau khi đã trích xuất xong keyframes để giải phóng ổ cứng Colab
    shutil.rmtree(temp_videos, ignore_errors=True)
    os.makedirs(temp_videos, exist_ok=True)
    print(f"[{datetime.now().strftime('%H:%M:%S')}] [✓] Đã hoàn thành keyframes cho {filename} & dọn sạch video tạm!")

# Thống kê tổng hợp kết quả keyframes
kf_dir = Path("/content/workspace/data/keyframes")
all_video_dirs = sorted([d.name for d in kf_dir.iterdir() if d.is_dir()])
total_keyframes = sum(1 for _ in kf_dir.rglob("*.jpg"))
print("\n" + "=" * 65)
print(f"🎉 HOÀN THÀNH TOÀN BỘ KEYFRAMES (-min 1.0) CHO 2 BATCH!")
print(f"   - Tổng số video đã xử lý: {len(all_video_dirs)} video")
print(f"   - Tổng số ảnh keyframe trích xuất: {total_keyframes:,} ảnh")
print(f"   - Danh sách video mẫu: {all_video_dirs[:8]} ... {all_video_dirs[-5:]}")

### 5. Sao Lưu Toàn Bộ Keyframes Sang Google Drive

- Đóng gói toàn bộ thư mục `data/keyframes/` thành `keyframes_N081_N100.tar.gz` lưu trực tiếp trên Drive.
- Dùng để tải về máy local chạy OCR bằng GPU RTX 4060 hoặc đối soát kết quả tìm kiếm.

In [ ]:
import os
import time
import subprocess
from datetime import datetime

drive_backup = "/content/drive/MyDrive/Vecna_N081_N100_Output"
os.makedirs(drive_backup, exist_ok=True)
kf_tar = os.path.join(drive_backup, "keyframes_N081_N100.tar.gz")

print(f"[{datetime.now().strftime('%H:%M:%S')}] 📦 Đang đóng gói và lưu keyframes lên Google Drive...")
t0 = time.time()
subprocess.run(["tar", "-czf", kf_tar, "-C", "/content/workspace/data", "keyframes"], check=True)
elapsed = time.time() - t0
size_mb = os.path.getsize(kf_tar) / (1024 * 1024)

print(f"[{datetime.now().strftime('%H:%M:%S')}] [✓] Đã lưu thành công keyframes lên Drive: {kf_tar}")
print(f"   - Dung lượng file: {size_mb:.2f} MB")
print(f"   - Thời gian nén: {elapsed:.1f} giây")

### 6. Trích Xuất Vector Ngữ Nghĩa Qwen-VL Embedding (`qwen_vl.npy`)

- Sử dụng mô hình `Qwen/Qwen3-VL-Embedding-2B` (Vector 2048 chiều).
- Hiển thị log thời gian thực chi tiết (docs/s, tiến độ từng video, ETA).

In [ ]:
import os
import sys
import time
import subprocess
from datetime import datetime

workspace_dir = "/content/workspace"
start_time = time.time()

def get_time_str():
    return datetime.now().strftime("%H:%M:%S")

def format_elapsed(seconds):
    mins, secs = divmod(int(seconds), 60)
    hours, mins = divmod(mins, 60)
    if hours > 0:
        return f"{hours:02d}h {mins:02d}m {secs:02d}s"
    return f"{mins:02d}m {secs:02d}s"

cmd = ["aic51-cli", "analyse", "--use-qwen-vl", "--keep-going"]
print(f"[{get_time_str()}] 🚀 BẮT ĐẦU TRÍCH XUẤT QWEN-VL: {' '.join(cmd)}")
print("-" * 70)

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

process = subprocess.Popen(
    cmd,
    cwd=workspace_dir,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env
)

for line in iter(process.stdout.readline, ""):
    line_clean = line.strip()
    if line_clean:
        elapsed = time.time() - start_time
        print(f"[{get_time_str()} | +{format_elapsed(elapsed)}] {line_clean}")
        sys.stdout.flush()

process.stdout.close()
return_code = process.wait()

total_time = time.time() - start_time
print("-" * 70)
if return_code == 0:
    print(f"[{get_time_str()}] [✓] HOÀN TẤT QWEN-VL! Tổng thời gian chạy: {format_elapsed(total_time)}")
else:
    print(f"[{get_time_str()}] [!] Quá trình dừng với mã {return_code}. Tổng thời gian: {format_elapsed(total_time)}")

### 7. Sao Lưu Features Qwen Lên Drive & Kiểm Tra Tính Toàn Vẹn

- Đóng gói toàn bộ `features/` thành `features_qwen_N081_N100.tar.gz` lưu sang Google Drive.
- Kiểm tra shape vector và dtype của các file numpy mẫu.

In [ ]:
import os
import time
import subprocess
import numpy as np
from pathlib import Path
from datetime import datetime

workspace_dir = "/content/workspace"
drive_backup = "/content/drive/MyDrive/Vecna_N081_N100_Output"
features_dir = Path("/content/workspace/features")
features_tar = os.path.join(drive_backup, "features_qwen_N081_N100.tar.gz")

print(f"[{datetime.now().strftime('%H:%M:%S')}] 📦 Đang đóng gói vector Qwen lên Google Drive...")
qwen_files = list(features_dir.rglob("qwen_vl.npy"))
video_dirs = [d for d in features_dir.iterdir() if d.is_dir()]

print(f"Thống kê trích xuất:")
print(f"   - Tổng số video có feature: {len(video_dirs)}")
print(f"   - Tổng số vector qwen_vl.npy: {len(qwen_files):,} file")

t0 = time.time()
subprocess.run(["tar", "-czf", features_tar, "-C", workspace_dir, "features"], check=True)
elapsed = time.time() - t0
size_mb = os.path.getsize(features_tar) / (1024 * 1024)

print(f"\n[✓] ĐÃ LƯU THÀNH CÔNG LÊN GOOGLE DRIVE!")
print(f"   - Đường dẫn: {features_tar}")
print(f"   - Dung lượng: {size_mb:.2f} MB")
print(f"   - Thời gian nén: {elapsed:.1f} giây")

# Đọc thử một số vector ngẫu nhiên để xác nhận
print("\n🔍 Kiểm tra dữ liệu mẫu:")
for sample in qwen_files[:3]:
    vec = np.load(sample)
    print(f"   - File: {sample.relative_to(features_dir)} | Shape: {vec.shape} | Dtype: {vec.dtype}")

### 8. Tóm Tắt & Cách Tải Về Máy Local Chạy OCR

Sau khi Colab hoàn thành, trên Google Drive (`MyDrive/Vecna_N081_N100_Output/`) bạn sẽ có 2 file:
1. `keyframes_N081_N100.tar.gz` (toàn bộ ảnh keyframe của 60 video)
2. `features_qwen_N081_N100.tar.gz` (toàn bộ vector Qwen)

**Để đưa vào máy Local chạy OCR bằng GPU RTX 4060:**
```powershell
# 1. Giải nén keyframes vào workspace local (ví dụ workspace_3):
tar -xzf keyframes_N081_N100.tar.gz -C e:\Projects\Vecna\workspace_3\data

# 2. Giải nén Qwen features vào workspace local:
tar -xzf features_qwen_N081_N100.tar.gz -C e:\Projects\Vecna\workspace_3

# 3. Kích hoạt môi trường và chạy OCR local:
cd e:\Projects\Vecna\workspace_3
..\.venv\Scripts\aic51-cli.exe analyse --use-ocr --keep-going
```